# 🔬 SRΨ-Engine: Physical Dimension Tests (v2 - Fresh Start)

**目标**: 验证 TRAE 的洞察 - "智能即物理学，不变性 > 拟合度"

> **"Conv 的低 Loss 是'虚假胜利'"**
> **"SRΨ 的高 Loss 是'物理代价'"**

---

## 📋 测试计划 (Phase 1A + 1C + 1B + 2)

### **Phase 1A: 压力测试** (核心)
- **Shift Robustness**: s ∈ {0, 4, 16, 32, 64} ← 半周期绝杀
- **Long-Range Rollout**: 100 步，分段评估
- **Noise + Recovery**: σ=0.15，41-60 步恢复测试
- **动量守恒**: Burgers 方程的守恒量验证

### **Phase 1C: 零样本外推** (关键)
- **T=100 超长预测** (2 倍训练长度)

### **Phase 1B: 极端物理场景** (重要)
- **Shock Wave**: 强激波测试
- **Multi-Peak**: 多波干涉测试

### **Phase 2: 谱分析** (高级)
- **傅里叶频谱分析**: 能量级联验证

---

## ⏱️ 预计时间: ~30-35 分钟 (A100 GPU)

## 🎯 物理性评分体系

| 维度 | 权重 | 说明 |
|------|------|----------|
| **不变性** | 40% | Shift=64 的误差增长率 |
| **守恒性** | 30% | 能量 + 动量守恒 |
| **韧性** | 20% | 噪声恢复 + 极端场景 |
| **拟合度** | 10% | 基础 MSE |

---

## 📝 使用说明

**运行方式**:
1. 点击每个代码块左侧的 ▶️ 按钮
2. **按顺序执行**（从上到下）
3. 等待每个步骤完成后再运行下一个

**重要**:
- ⚠️ **不要跳过任何步骤**
- ⚠️ **上传所有 4 个 checkpoint 文件**
- ⚠️ **保持标签页开启**（已添加防休眠脚本）

## Step 0: 环境检查与清理

**重要**: 确保干净的起始状态

In [ ]:
# 清理可能存在的旧嵌套目录
import os

print('🔍 Checking for nested directories...')

# 检查是否在 srpsi-engine-tiny 中
if os.path.exists('/content/srpsi-engine-tiny'):
    # 如果已经在 srpsi-engine-tiny 中，检查是否有嵌套
    nested = os.path.exists('/content/srpsi-engine-tiny/srpsi-engine-tiny')
    if nested:
        print('❌ Found nested directories! Cleaning...')
        os.system('cd /content && rm -rf srpsi-engine-tiny')
        print('✅ Old nested directories removed')
    else:
        print('✅ Directory structure OK')
        
# 检查当前目录
print(f'\n📍 Current directory: {os.getcwd()}')

## Step 1: 克隆仓库并同步最新代码

In [ ]:
print('\n' + '='*70)
print(' ' * 20 + 'REPOSITORY SETUP')
print('='*70)

# Clone repository
print('\n📥 Cloning repository...')
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git

# Change to project directory
%cd srpsi-engine-tiny

# ⭐ CRITICAL: Pull latest updates
print('\n🔄 Pulling latest updates...')
!git pull origin main

# Verify directory structure
print('\n🔍 Verifying directory structure...')
!pwd
!ls -la

print('\n✅ Setup complete!')

## Step 2: 检查 GPU 并安装依赖

In [ ]:
import torch
import numpy as np
from pathlib import Path
import json
import sys
from datetime import datetime

print('\n' + '='*70)
print(' ' * 20 + 'GPU CHECK & SETUP')
print('='*70)

# Check GPU
if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f'\n✅ GPU Available: {gpu_name}')
    print(f'   Memory: {gpu_memory:.2f} GB')
    
    # Check if A100
    if 'A100' in gpu_name:
        print(f'   ✅ A100 detected! Optimizations enabled.')
        optimize_for_a100 = True
    else:
        print(f'   ⚠️  Using {gpu_name}, optimizations still enabled.')
        optimize_for_a100 = True
else:
    device = 'cpu'
    print('⚠️  No GPU found, using CPU (will be slow)')
    optimize_for_a100 = False

print(f'\n📍 Device: {device}')
print(f'🚀 A100 Optimization: {"Enabled" if optimize_for_a100 else "Disabled"}')

# Install dependencies
print('\n📦 Installing dependencies...')
!pip install -q pyyaml matplotlib

print('\n✅ Dependencies installed')

# Add src to path
sys.path.insert(0, 'src')
print('✅ src/ added to Python path')

## Step 3: 上传 Checkpoint 文件

**重要**: 需要上传 4 个 checkpoint 文件到 `checkpoints_colab/` 目录

### 文件列表:

1. **exp4_conv_final.pt** (Conv Baseline)
2. **exp5_transformer_final.pt** (Transformer)
3. **srpsi_real_final.pt** (SRΨ Real from Windows TRAE)
4. **srpsi_no_r_final.pt** (SRΨ w/o R from Windows TRAE)

In [ ]:
print('\n' + '='*70)
print(' ' * 23 + 'CHECKPOINT UPLOAD')
print('='*70)

# Create checkpoints directory
ckpt_dir = Path('checkpoints_colab')
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Required files
required_files = {
    'exp4_conv_final.pt': 'Conv Baseline (Colab)',
    'exp5_transformer_final.pt': 'Transformer (Colab)',
    'srpsi_real_final.pt': 'SRΨ Real (Windows TRAE)',
    'srpsi_no_r_final.pt': 'SRΨ w/o R (Windows TRAE)'
}

print('\n📤 Required checkpoint files:\n')
for f, desc in required_files.items():
    print(f'  • {f}')
    print(f'    Source: {desc}')

print('\n📊 Current status:\n')
all_ready = True
for f in required_files.keys():
    exists = (ckpt_dir / f).exists()
    status = '✅' if exists else '❌'
    size = f"({(ckpt_dir / f).stat().st_size / 1e6:.1f} MB)" if exists else ""
    print(f'  {status} {f} {size}')
    if not exists:
        all_ready = False

if all_ready:
    print('\n✅ All checkpoints ready!')
    print('   ⏭️  Ready to proceed to Step 4')
else:
    print('\n⏳ Please upload missing checkpoints:')
    print('   1. Click 📁 icon on the left')
    print('   2. Navigate to srpsi-engine-tiny/checkpoints_colab/')
    print('   3. Click upload button (📤)')
    print('   4. Select missing files')
    print('\n   Then run this cell again to verify.')

## Step 4: 生成测试数据

In [ ]:
print('\n' + '='*70)
print(' ' * 25 + 'DATA GENERATION')
print('='*70)

import os
os.makedirs('data', exist_ok=True)

print('\n⏳ Generating test data...')
print('   (This may take 1-2 minutes)\n')

!python src/data_gen.py

print('\n✅ Data generation complete')

# Verify data
data_path = Path('data/burgers_1d.npy')
if data_path.exists():
    print('\n🔍 Verifying data...')
    data_loaded = np.load(data_path)
    print(f'   Data shape: {data_loaded.shape}')
    print(f'   Data type: {data_loaded.dtype}')
    print('\n✅ Data ready!')
else:
    print('❌ Data file not found!')

## Step 5: 加载模型（使用内联工厂函数）

**关键改进**: 直接在 notebook 中定义模型工厂函数，避免 import 问题

In [ ]:
print('\n' + '='*70)
print(' ' * 23 + 'MODEL LOADING (A100)')
print('='*70)

# Import required modules
from src.models import (
    ConvBaseline,
    TransformerRelPE,
    SRPsiEngineReal,
    SRPsiEngineNoR
)
from src.utils import load_config

print('\n✅ Model classes imported successfully')

# Define model factory function inline
def create_model_inline(model_type: str, cfg: dict, device: torch.device):
    """Create model instance based on type and config."""
    tin = cfg['task']['tin']
    tout = cfg['task']['tout']
    nx = cfg['task']['nx']
    hidden_dim = cfg['model']['hidden_dim']
    depth = cfg['model']['depth']
    kernel_size = cfg['model']['kernel_size']

    if model_type == 'conv_baseline':
        model = ConvBaseline(
            tin=tin, nx=nx,
            hidden_dim=hidden_dim,
            depth=depth,
            kernel_size=kernel_size,
            tout=tout
        )
    elif model_type == 'transformer_rel_pe':
        model = TransformerRelPE(
            tin=tin, nx=nx,
            d_model=hidden_dim,
            nhead=4,
            num_layers=depth,
            dropout=cfg['model']['dropout'],
            tout=tout
        )
    elif model_type == 'srpsi_real':
        model = SRPsiEngineReal(
            tin=tin, nx=nx,
            hidden_dim=hidden_dim,
            depth=depth,
            kernel_size=kernel_size,
            dt=0.01,
            tout=tout
        )
    elif model_type == 'srpsi_no_r':
        model = SRPsiEngineNoR(
            tin=tin, nx=nx,
            hidden_dim=hidden_dim,
            depth=depth,
            kernel_size=kernel_size,
            dt=0.01,
            tout=tout
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model.to(device)

print('✅ Model factory function defined')

# Data loader helper
def load_burgers_data():
    """Smart data loader for numpy array format."""
    data_loaded = np.load('data/burgers_1d.npy')
    
    # Handle [num_samples, total_steps, nx] format
    num_samples, total_steps, nx = data_loaded.shape
    
    # Split 80/10/10
    n_train = int(0.8 * num_samples)
    n_val = int(0.1 * num_samples)
    n_test = num_samples - n_train - n_val
    
    u_all = data_loaded
    
    return {
        'u_train': u_all[:n_train],
        'u_val': u_all[n_train:n_train+n_val],
        'u_test': u_all[n_train+n_val:],
        'nu_train': np.full(n_train, 0.01, dtype=np.float32),
        'nu_val': np.full(n_val, 0.01, dtype=np.float32),
        'nu_test': np.full(n_test, 0.01, dtype=np.float32),
        'x': np.linspace(0, 2*np.pi, nx, endpoint=False, dtype=np.float32)
    }

print('✅ Data loader helper defined')

# Model configurations
model_configs = [
    {
        'name': 'Exp4_Conv',
        'ckpt': 'checkpoints_colab/exp4_conv_final.pt',
        'type': 'conv_baseline',
        'config': 'config/burgers.yaml'
    },
    {
        'name': 'Exp5_Transformer',
        'ckpt': 'checkpoints_colab/exp5_transformer_final.pt',
        'type': 'transformer_rel_pe',
        'config': 'config/burgers.yaml'
    },
    {
        'name': 'Exp2_SRΨ_Real',
        'ckpt': 'checkpoints_colab/srpsi_real_final.pt',
        'type': 'srpsi_real',
        'config': 'config/burgers.yaml'
    },
    {
        'name': 'Exp3_SRΨ_w/o_R',
        'ckpt': 'checkpoints_colab/srpsi_no_r_final.pt',
        'type': 'srpsi_no_r',
        'config': 'config/burgers.yaml'
    }
]

print('\n🔄 Loading models...\n')

# Load all models
models = {}
total_params = 0

for config in model_configs:
    print(f'📦 Loading {config["name"]}...')
    
    ckpt_path = Path(config['ckpt'])
    if not ckpt_path.exists():
        print(f'  ❌ Checkpoint not found: {config["ckpt"]}')
        print(f'     Full path: {ckpt_path.absolute()}')
        continue
    
    try:
        # Load configuration
        cfg = load_config(config['config'])
        
        # Load checkpoint
        ckpt = torch.load(config['ckpt'], map_location=device)
        
        # Create model instance using inline factory
        model = create_model_inline(config['type'], cfg, device)
        
        # Load state dict
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        
        # Count parameters
        n_params = sum(p.numel() for p in model.parameters())
        total_params += n_params
        
        # Store model
        models[config['name']] = {
            'model': model,
            'type': config['type'],
            'train_loss': float(ckpt.get('loss', 0)),
            'epoch': int(ckpt.get('epoch', 0)),
            'params': n_params
        }
        
        print(f'  ✅ Loaded successfully')
        print(f'     - Epoch: {models[config["name"]]["epoch"]}/80')
        print(f'     - Train Loss: {models[config["name"]]["train_loss"]:.2f}')
        print(f'     - Parameters: {n_params:,}')
        
    except Exception as e:
        print(f'  ❌ Failed to load: {e}')
        import traceback
        traceback.print_exc()
        continue

print('\n' + '='*70)
print(f'✅ Successfully loaded {len(models)}/4 models')
print(f'📊 Total parameters: {total_params:,}')

if device == 'cuda':
    memory_used = torch.cuda.memory_allocated() / 1e9
    memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🎮 GPU Memory: {memory_used:.2f} GB / {memory_total:.2f} GB')
    print(f'   Available: {memory_total - memory_used:.2f} GB')

if len(models) < 4:
    print('\n⚠️  Warning: Some models failed to load!')
    print('   Please check checkpoint files and try again.')
else:
    print('\n✅ All models ready for testing!')

## Step 6: Phase 1A - Shift Robustness Test

**测试**: s ∈ {0, 4, 16, 32, 64}

**预计时间**: 5-8 分钟

In [ ]:
print('\n' + '='*70)
print(' ' * 15 + 'PHASE 1A: SHIFT ROBUSTNESS TEST')
print('='*70)

print('\n🎯 Testing: Spatial shift invariance')
print('   Hypothesis: Conv error explodes at shift=64 (half-period)')
print('            SRΨ error remains stable\n')

# Load test data
data = load_burgers_data()
u_init = torch.tensor(data['u_test'][:100], dtype=torch.float32)
u_true = torch.tensor(data['u_test'][:100, 10:], dtype=torch.float32)

shifts = [0, 4, 16, 32, 64]
results_shift = {name: [] for name in models.keys()}

print('📊 Testing shifts:', shifts)
print('📊 Samples: 100\n')

for shift in shifts:
    print(f'📍 Shift = {shift}...', end=' ', flush=True)
    
    # Shift input (circular roll)
    u_init_shifted = torch.roll(u_init, shift, dims=-1)
    
    for name, model_info in models.items():
        model = model_info['model']
        
        with torch.no_grad():
            u_pred = model(u_init_shifted.to(device))
            u_pred = u_pred.cpu()
        
        # Compute MSE
        mse = torch.mean((u_pred - u_true) ** 2).item()
        results_shift[name].append(mse)
    
    print('✅')

print('\n' + '-'*70)
print(' ' * 25 + 'RESULTS')
print('-'*70)

print(f'\n{"Model":<20} {"S=0":<10} {"S=4":<10} {"S=16":<10} {"S=32":<10} {"S=64":<10} {"Growth":<10}')
print('-'*80)

for name in models.keys():
    errors = results_shift[name]
    growth = errors[-1] / (errors[0] + 1e-10)
    print(f'{name:<20} {errors[0]:<10.2f} {errors[1]:<10.2f} {errors[2]:<10.2f} {errors[3]:<10.2f} {errors[4]:<10.2f} {growth:<10.2f}x')

print('\n💡 Key insight: Lower growth rate = better shift invariance')
print('   Conv expected to have high growth at shift=64')

## Step 7: Phase 1A - Energy & Momentum Test

**测试**: 100-step rollout

**预计时间**: 4-6 分钟

In [ ]:
print('\n' + '='*70)
print(' ' * 15 + 'PHASE 1A: ENERGY & MOMENTUM TEST')
print('='*70)

print('\n🎯 Testing: 100-step rollout with conservation metrics')
print('   Hypothesis: SRΨ maintains better energy/momentum conservation\n')

rollout_steps = 100
test_samples = 10
results_energy = {name: {'energy': [], 'momentum': []} for name in models.keys()}

print(f'📊 Rollout steps: {rollout_steps}')
print(f'📊 Samples: {test_samples}\n')

data = load_burgers_data()

for name, model_info in models.items():
    print(f'🔄 Testing {name}...', end=' ', flush=True)
    
    model = model_info['model']
    u_current = data['u_test'][:test_samples, :10]
    u_current = torch.tensor(u_current, dtype=torch.float32)
    
    energy_history = []
    momentum_history = []
    
    for step in range(rollout_steps):
        with torch.no_grad():
            u_next = model(u_current.to(device)).cpu()
        
        # Compute energy (L2 norm)
        energy = torch.sqrt(torch.sum(u_next ** 2, dim=(-1,))).mean().item()
        energy_history.append(energy)
        
        # Compute momentum (sum over space)
        momentum = torch.sum(u_next, dim=(-1,)).mean().item()
        momentum_history.append(momentum)
        
        # Auto-regressive: shift window
        u_current = torch.cat([u_current[:, 1:], u_next[:, -1:]], dim=1)
    
    results_energy[name]['energy'] = energy_history
    results_energy[name]['momentum'] = momentum_history
    
    print('✅')

print('\n' + '-'*70)
print(' ' * 25 + 'RESULTS')
print('-'*70)

print(f'\n{"Model":<20} {"E_Drift":<12} {"M_Drift":<12} {"E_Monotonic":<15}')
print('-'*70)

for name in models.keys():
    energy = results_energy[name]['energy']
    momentum = results_energy[name]['momentum']
    
    # Drift: (final - initial) / initial
    e_drift = abs(energy[-1] - energy[0]) / (abs(energy[0]) + 1e-10)
    m_drift = abs(momentum[-1] - momentum[0]) / (abs(momentum[0]) + 1e-10)
    
    # Monotonicity
    e_changes = [abs(energy[i+1] - energy[i]) for i in range(len(energy)-1)]
    e_max_change = max(e_changes) / (abs(energy[0]) + 1e-10)
    
    monotonic = "Stable" if e_max_change < 0.1 else "Fluctuating"
    
    print(f'{name:<20} {e_drift:<12.4f} {m_drift:<12.4f} {monotonic:<15}')

print('\n💡 Key insight: Lower drift + stable monotonicity = better physics')

## Step 8: Phase 1A - Noise + Recovery Test

**测试**: σ=0.15 高斯噪声

**预计时间**: 3-5 分钟

In [ ]:
print('\n' + '='*70)
print(' ' * 15 + 'PHASE 1A: NOISE + RECOVERY TEST')
print('='*70)

print('\n🎯 Testing: Field self-recovery after noise removal')
print('   Hypothesis: SRΨ recovers via Ψ stabilizer\n')

noise_std = 0.15
test_samples = 100
results_noise = {name: [] for name in models.keys()}

print(f'📊 Noise std: {noise_std}')
print(f'📊 Samples: {test_samples}\n')

data = load_burgers_data()

# Test with noise
print('📊 Phase 1: With noise...', end=' ', flush=True)
u_init = torch.tensor(data['u_test'][:test_samples, :10], dtype=torch.float32)
u_true = torch.tensor(data['u_test'][:test_samples, 10:], dtype=torch.float32)

# Add noise
u_noisy = u_init + torch.randn_like(u_init) * noise_std

for name, model_info in models.items():
    model = model_info['model']
    
    with torch.no_grad():
        u_pred = model(u_noisy.to(device)).cpu()
    
    mse = torch.mean((u_pred - u_true) ** 2).item()
    results_noise[name].append(mse)

print('✅')

# Test recovery (no noise)
print('📊 Phase 2: After recovery (no noise)...', end=' ', flush=True)

for name, model_info in models.items():
    model = model_info['model']
    
    with torch.no_grad():
        u_pred = model(u_init.to(device)).cpu()
    
    mse = torch.mean((u_pred - u_true) ** 2).item()
    results_noise[name].append(mse)

print('✅')

print('\n' + '-'*70)
print(' ' * 25 + 'RESULTS')
print('-'*70)

print(f'\n{"Model":<20} {"With Noise":<15} {"After Recovery":<18} {"Recovery":<10}')
print('-'*70)

for name in models.keys():
    errors = results_noise[name]
    recovery_rate = errors[0] / (errors[1] + 1e-10)
    print(f'{name:<20} {errors[0]:<15.2f} {errors[1]:<18.2f} {recovery_rate:<10.2f}x')

print('\n💡 Key insight: Lower recovery rate = better self-recovery')
print('   SRΨ expected to have recovery rate closer to 1.0')

## Step 9: 保存结果

**保存所有测试结果到 JSON 文件**

In [ ]:
print('\n' + '='*70)
print(' ' * 22 + 'SAVE RESULTS')
print('='*70)

# Prepare results
results = {
    'timestamp': datetime.now().isoformat(),
    'device': device,
    'models': {
        name: {
            'train_loss': info['train_loss'],
            'epoch': info['epoch'],
            'parameters': info['params']
        }
        for name, info in models.items()
    },
    'phase_1a': {
        'shift_robustness': {
            'shifts': shifts,
            'results': results_shift
        },
        'energy_momentum': results_energy,
        'noise_recovery': results_noise
    }
}

# Save to JSON
Path('results').mkdir(exist_ok=True)
output_path = 'results/physical_dimension_tests_v2.json'

with open(output_path, 'w') as f:
    json.dump(results, f, indent=2, default=lambda x: float(x) if isinstance(x, np.float64) else x)

print(f'\n✅ Results saved to: {output_path}')
print(f'\n📥 Download instructions:')
print(f'   1. Click 📁 icon on the left')
print(f'   2. Navigate to srpsi-engine-tiny/results/')
print(f'   3. Download {output_path}')

print('\n' + '='*70)
print('✅ Phase 1A tests completed!')
print('='*70)

## 🎉 Phase 1A 完成！

---

### **已完成测试**

1. ✅ **Shift Robustness** (半周期绝杀)
2. ✅ **Energy & Momentum Conservation** (100-step rollout)
3. ✅ **Noise Recovery** (σ=0.15)

---

### **关键洞察**

> **"Conv 的低 Loss 是'虚假胜利'"**
> **"SRΨ 的高 Loss 是'物理代价'"**
> **"智能体现在不变性，而非拟合度"**

---

### **下一步**

1. **下载结果** (JSON 文件)
2. **分析数据**，验证 TRAE 的假说
3. **继续 Phase 1C/1B/2** (如果需要)

---

**感谢参与这次物理本质的探索！** 🚀